In [0]:
dbutils.library.restartPython()


In [0]:
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_jdbc, write_raw
import json

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("sql-server-ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================
dbutils.widgets.text("path", "")
config_path =  dbutils.widgets.get("path")

# ============================================================
# 2. READ CONFIG
# ============================================================
with open(f"{config_path}", "r") as f:
    config = json.load(f)


# ============================================================
# 3. CONFIG SECTIONS
# ============================================================
source_config = config['source']
target_config = config['target']
writer_config = config['write_options']


# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat()
logger.info("Load date is %s", run_date)
logger.info("Reading data from %s", source_config['dbtable'])


user = dbutils.secrets.get(scope='retail-platform-dev', key=source_config['user'])
password = dbutils.secrets.get(scope='retail-platform-dev', key=source_config['password'])

# ============================================================
# 5. READ FROM SQL SERVER
# ============================================================
df = read_jdbc(spark, source_config['url'], source_config['dbtable'], user, password)

# ============================================================
# 6. TARGET PATH
# ============================================================
target_path = f"{target_config['base_path']}/{target_config['folder']}/load_date={run_date}"
logger.warning("Writing data at target path %s", target_path)
# ============================================================
# 7. WRITE RAW
# ============================================================
write_raw(df, target_path, target_config['file_format'], target_config['mode'], writer_config)
logger.info("Data Written Succesfully")


In [0]:
"""
%sql
create schema if not exists retaildataplatform.bronze
"""


In [0]:
"""
%sql
create volume if not exists retaildataplatform.bronze.raw_data
"""